# Freshness Detection Model Training

Run this notebook in Google Colab with GPU enabled (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!pip install kaggle tensorflow scikit-learn

## Setup Kaggle API
Paste your `KAGGLE_API_TOKEN` (starts with `KGAT_...`) to download the datasets directly to Colab.

In [ ]:
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Enter your KAGGLE_API_TOKEN: ')

## Download Dataset

In [ ]:
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification
!unzip -q fruits-fresh-and-rotten-for-classification.zip -d dataset
!ls dataset

## Train Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Setup paths and data generators
train_dir = "dataset/dataset/train"
test_dir = "dataset/dataset/test"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Build Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
history = model.fit(train_generator, validation_data=val_generator, epochs=5)

# Save Model
model.save("freshness_model.h5")


## Download the Trained Model
After training, you can download `freshness_model.h5` and place it in the `nutrilens-ai/ml_models` folder in your project.

In [ ]:
files.download('freshness_model.h5')